# 04 · 因子组合研究

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next((candidate for candidate in (start, *start.parents) if (candidate / 'vbt').exists()), start)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'项目根目录: {ROOT}')

## 1. 加载因子与未来收益

In [ ]:
from itertools import combinations
import numpy as np
import pandas as pd
from vbt.adapters import DEFAULT_FACTORS, VBTDataLoader
data = VBTDataLoader(start_date='2020-01-01', end_date='2024-12-31').load(factors=DEFAULT_FACTORS)
future_return = data['close'].pct_change(20).shift(-20)
candidate_factors = list(DEFAULT_FACTORS)
candidate_factors

## 2. 单因子 Rank IC 与相关性

In [ ]:
ic = pd.DataFrame({name: data[name].corrwith(future_return, axis=1, method='spearman') for name in candidate_factors})
summary = pd.DataFrame({'IC均值': ic.mean(), 'IC标准差': ic.std(), 'ICIR': ic.mean() / ic.std()}).sort_values('ICIR', ascending=False)
display(summary)
latest = pd.DataFrame({name: data[name].iloc[-1] for name in candidate_factors})
display(latest.corr(method='spearman'))

## 3. 遍历二/三因子等权 Rank 组合

In [ ]:
rows = []
for size in (2, 3):
    for names in combinations(candidate_factors, size):
        daily_ic = ic[list(names)].mean(axis=1)
        rows.append({'factors': '+'.join(names), 'IC': daily_ic.mean(), 'ICIR': daily_ic.mean() / daily_ic.std()})
ranking = pd.DataFrame(rows).sort_values('ICIR', ascending=False)
display(ranking.head(20))

## 4. 导出组合排名

In [ ]:
from datetime import datetime
path = ROOT / 'output/vectorbt/param_scans' / f"factor_combinations_{datetime.now():%Y%m%d_%H%M%S}.parquet"
path.parent.mkdir(parents=True, exist_ok=True)
ranking.to_parquet(path, index=False)
path